In [1]:
%pip install pandas xarray netCDF4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# CELL 1 — Load CPCB + ERA5
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CPCB_PATH = ROOT / "data/intermediate/cpcb_master_2021_2025.csv"
ERA5_PATH = ROOT / "data/intermediate/era5_merged_2021_2025.nc"
OUTPUT = ROOT / "data/processed/cpcb_era5_2021_2025.csv"

cpcb = pd.read_csv(CPCB_PATH, parse_dates=["datetime_utc"])
ds = xr.open_dataset(ERA5_PATH, engine="netcdf4")

print("CPCB :", cpcb.shape)
print("Stations:", cpcb.location_id.nunique())
print("ERA5 :", dict(ds.sizes))
print("Variables:", list(ds.data_vars))

CPCB : (1621488, 14)
Stations: 37
ERA5 : {'valid_time': 14608, 'latitude': 6, 'longitude': 7}
Variables: ['u10', 'v10', 'd2m', 't2m', 'sp', 'tp', 'blh']


In [3]:
# ============================================================
# CELL 2 — Coverage & ERA5 Preprocessing
# ============================================================

time_col = "valid_time" if "valid_time" in ds.coords else "time"
era5_t = pd.to_datetime(ds[time_col].values)

print(f"CPCB: {cpcb.datetime_utc.min()} → {cpcb.datetime_utc.max()}")
print(f"ERA5: {era5_t.min()} → {era5_t.max()}")

era5 = ds.copy()
era5["t2m_c"] = era5["t2m"] - 273.15
era5["d2m_c"] = era5["d2m"] - 273.15
era5["sp_hpa"] = era5["sp"] / 100
era5["tp_mm"] = era5["tp"] * 1000

print("ERA5 preprocessing complete.")

CPCB: 2021-01-01 00:00:00 → 2025-12-31 23:00:00
ERA5: 2021-01-01 00:00:00 → 2025-12-31 21:00:00
ERA5 preprocessing complete.


In [4]:
# ============================================================
# CELL 3 — ERA5 Station Interpolation
# ============================================================

station_meta = cpcb[
    ["location_id","station_name","latitude","longitude"]
].drop_duplicates("location_id")

valid_times = np.sort(
    cpcb.loc[
        cpcb.datetime_utc.between(era5_t.min(), era5_t.max()),
        "datetime_utc"
    ].unique()
)

frames = []

for _, s in station_meta.iterrows():
    point = era5.interp(
        latitude=s.latitude,
        longitude=s.longitude,
        method="linear"
    ).interp(
        {time_col: valid_times},
        method="linear"
    )

    d = point[
        ["t2m_c","d2m_c","sp_hpa","tp_mm","u10","v10","blh"]
    ].to_dataframe().reset_index()

    d = d.rename(columns={time_col: "datetime_utc"})
    d["location_id"] = s.location_id
    frames.append(d)

era5_station = pd.concat(frames, ignore_index=True)

era5_station["era5_wind_speed"] = np.hypot(
    era5_station.u10, era5_station.v10
)

era5_station["era5_wind_dir"] = (
    180 + np.degrees(
        np.arctan2(era5_station.u10, era5_station.v10)
    )
) % 360

era5_station = era5_station.rename(columns={
    "t2m_c": "era5_t2m_c",
    "d2m_c": "era5_d2m_c",
    "sp_hpa": "era5_sp_hpa",
    "tp_mm": "era5_tp_mm",
    "blh": "era5_blh"
})[
    [
        "location_id","datetime_utc",
        "era5_t2m_c","era5_d2m_c","era5_sp_hpa",
        "era5_tp_mm","era5_wind_speed",
        "era5_wind_dir","era5_blh"
    ]
]

print(f"ERA5 station records: {len(era5_station):,}")

ERA5 station records: 1,621,414


In [5]:
# ============================================================
# CELL 4 — CPCB + ERA5 Merge & Validation
# ============================================================

era5_min, era5_max = era5_t.min(), era5_t.max()

merged = cpcb[
    cpcb.datetime_utc.between(era5_min, era5_max)
].merge(
    era5_station,
    on=["location_id","datetime_utc"],
    how="left",
    validate="one_to_one"
)

era5_cols = [
    "era5_t2m_c","era5_d2m_c","era5_sp_hpa",
    "era5_tp_mm","era5_wind_speed",
    "era5_wind_dir","era5_blh"
]

print("Original CPCB :", f"{len(cpcb):,}")
print("Merged        :", f"{len(merged):,}")
print("Excluded      :", f"{len(cpcb)-len(merged):,}")
print("Stations      :", merged.location_id.nunique())
print("Duplicate times:",
      merged.duplicated(["location_id","datetime_utc"]).sum())
print("Missing ERA5  :", f"{merged[era5_cols].isna().sum().sum():,}")
print("Date range    :",
      merged.datetime_utc.min(), "→", merged.datetime_utc.max())

print("\nMissing ERA5 by variable:")
print(merged[era5_cols].isna().sum())

Original CPCB : 1,621,488
Merged        : 1,621,414
Excluded      : 74
Stations      : 37
Duplicate times: 0
Missing ERA5  : 0
Date range    : 2021-01-01 00:00:00 → 2025-12-31 21:00:00

Missing ERA5 by variable:
era5_t2m_c         0
era5_d2m_c         0
era5_sp_hpa        0
era5_tp_mm         0
era5_wind_speed    0
era5_wind_dir      0
era5_blh           0
dtype: int64


In [6]:
# ============================================================
# CELL 5 — Save Final CPCB + ERA5 Dataset
# ============================================================

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)
print("Rows:", f"{len(merged):,}")
print("Stations:", merged.location_id.nunique())


Saved: c:\Users\udish\College\MajorProjectSem7\Implementation\data\processed\cpcb_era5_2021_2025.csv
Rows: 1,621,414
Stations: 37


In [7]:
# ============================================================
# CELL 6 — Final audit
# ============================================================

print("Shape:", merged.shape)
print("Columns:", len(merged.columns))
print("Missing total:", merged.isna().sum().sum())
print("Duplicate station-times:",
      merged.duplicated(["location_id","datetime_utc"]).sum())

Shape: (1621414, 21)
Columns: 21
Missing total: 1329368
Duplicate station-times: 0
